# 03 · Construcción de variables

**Objetivo.** Construir el desenlace y las exposiciones codificando el «no aplica» de forma explícita, y guardar el dataset único del proyecto.

| | |
|---|---|
| **Entradas** | DataFrame limpio |
| **Salidas** | `datos/final/data.parquet`, `tabla_00_diccionario`, `tabla_04_flujograma_strobe` |

> Proyecto: *Modelamiento epidemiológico de la depresión en la población peruana de 15 y
> más años (ENDES 2025)* · Epidemiología 2026-2 · UNMSM


In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parents[1] / "codigo" / "src"))
sys.path.insert(0, str(Path.cwd().parents[1] / "codigo" / "scripts"))

import numpy as np
import pandas as pd
from depresion import config as cfg, io as dio, viz
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
viz.aplicar_tema()
print("Proyecto:", cfg.RAIZ.name)

Todo este notebook se resume en un script reproducible. Se ejecuta aquí para poder inspeccionar los pasos intermedios.

In [ ]:
%run ../scripts/construir_dataset.py

In [ ]:
df = pd.read_parquet(cfg.DATASET_FINAL)
print(f"Dataset: {df.shape}")
print((cfg.DATOS_FINAL / "data_hash.txt").read_text())

## Reglas de construcción

Cada variable derivada guarda la regla exacta que la produjo. Es lo que permite replicar el dataset en R o SPSS.

In [ ]:
from depresion import variables as var
for nombre, regla in list(var.REGLAS.items())[:12]:
    print(f"\n{nombre}\n  {regla}")

### El caso del alcohol

El bloque tiene saltos del tipo «si responde Sí, salta la siguiente»: el criterio ya quedó satisfecho. Contar ítems en crudo penalizaría justamente a los casos más graves, así que se agrupan por criterio con lógica OR.

In [ ]:
print(var.REGLAS["n_criterios_dependencia"])
print()
print(df.alcohol_categoria.value_counts(dropna=False).to_string())

## Conclusiones

- Dataset de **33 569 × 71**, con hash SHA-256 para versionarlo.
- Población de análisis: **30 226** (`en_analisis == 1`).
- El flujograma STROBE documenta cada exclusión con su N y su motivo.
- `violencia_fisica` **no** se codifica como 0 para quien no tuvo pareja: es una
  subpoblación distinta, no un cero.
